# 3 · From the pore to a current

Whether a channel conducts is three questions, not one, and they are kept apart
here because a pore can be shut in more than one way:

1. **Is it wide enough?**
2. **Would water stay in it?** A pore can be geometrically open and still block,
   because a hydrophobic neck expels liquid water.
3. **What current would flow?**

Run `python -m piezo1.io.fetch` first — question 2 needs the CHAP hydration
grid as well as the structures.

In [ ]:
import numpy as np

from piezo1.config import STRUCTURE_DIR
from piezo1.core.structure import Structure
from piezo1.structure.pore import pore_profile
from piezo1.structure.protomers import protomer_blocks
from piezo1.structure.superpose import detect_c3_axis

st = Structure.from_file(STRUCTURE_DIR / "11ZC.cif")
blocks, _ = protomer_blocks(st)
profile = pore_profile(st, detect_c3_axis(blocks), step=1.0)

print(f"bottleneck radius : {profile.bottleneck_radius:.2f} A")
print(f"at z              : {profile.bottleneck_z:.1f} A")
print(f"constrictions     : {len(profile.constrictions())}")

## Why the probe is on a leash

The radius at each height is the largest sphere that fits — but "largest sphere
that fits" has no interior maximum in an open system. Without a constraint
tethering the probe near the conduction axis, the optimiser walks out of the
protein entirely and reports a radius of about 6,000 Å. That is a true maximum,
and completely useless.

`pore.leash` is a registered parameter, so you can see it, change it, and see
what it does to the answer.

In [ ]:
from piezo1.parameters import PARAMETERS, reset, set_value

print("leash  :", PARAMETERS.value("pore.leash"), "A")
print("step   :", PARAMETERS.value("pore.step"), "A")

set_value("pore.step", 2.0)
coarse = pore_profile(st, detect_c3_axis(blocks))
print(f"\nbottleneck at 2.0 A slices: {coarse.bottleneck_radius:.2f} A")
reset()                       # always put the registry back
print("restored:", PARAMETERS.value("pore.step"), "A")

## Would water stay in it?

Radius alone predicts the conducting state at AUROC 0.59. Radius *combined
with* the hydrophobicity of the lining reaches 0.91 (Rao et al. 2019). The
heuristic scores the pore against an MD-derived free-energy grid; above 0.55
the lining would dewet and the channel is shut whatever its radius.

The two ways of being shut are reported **separately**, because PIEZO1 has
structures that are one and not the other.

In [ ]:
from piezo1.analysis.hydration import load_grid, predict_wetting

for pdb in ("8YEZ", "11ZC"):
    entry = Structure.from_file(STRUCTURE_DIR / f"{pdb}.cif")
    eb, _ = protomer_blocks(entry)
    prof = pore_profile(entry, detect_c3_axis(eb), step=1.0)
    wet = predict_wetting(entry, prof, load_grid())
    print(f"{pdb}: score {wet.score:5.2f}  "
          f"hydrophobic gate {str(wet.hydrophobic_gate):5s}  "
          f"sterically occluded {str(wet.sterically_occluded):5s}")
    print(f"      {wet.verdict}")

## What current would flow?

A one-dimensional drift-diffusion calculation over the measured pore, with the
access resistance at each mouth. The potential is solved in the electroneutral
limit, because the Debye length here (5.7–8.1 Å) is larger than the pore radius
(3.3 Å) — the usual Poisson–Nernst–Planck assumption of a well-screened channel
simply does not hold, and the module says so rather than proceeding quietly.

In [ ]:
from piezo1.physics.permeation import (blocking_mechanisms, default_species,
                                       solve_pnp)

blocks_11zc, _ = protomer_blocks(st)
prof = pore_profile(st, detect_c3_axis(blocks_11zc), step=1.0)
wet = predict_wetting(st, prof, load_grid())

result = solve_pnp(prof, default_species())
print(f"current      : {result.current * 1e12:6.2f} pA")
print(f"conductance  : {result.conductance * 1e12:6.1f} pS   (published 25-30)")
print(f"converged    : {result.converged}")
print(f"blocked by   : {result.blocked_by}")
print(f"mechanisms   : {blocking_mechanisms(wet, prof.radius, default_species())}")

## The number that does not agree, and why that is reported

40.7 pS against a published 25–30 pS. This project does **not** tune it into
agreement, because two of the inputs have never been measured for PIEZO1: the
in-pore diffusivity and the effective ion radius. Across the plausible ranges
of **both**, the answer spans 16–94 pS, which contains the published value
comfortably. The cell below varies only the first of the two, so it shows a
narrower spread than that.

Agreement reached by choosing values inside that range would be fitting, not
prediction. So the disagreement stands, with the reason attached.

In [ ]:
from piezo1.analysis.uncertainty import parameter_range

def conductance_at(_scale):
    # parameter_range sets the registry key before each call and restores it
    # afterwards, even if the statistic raises - so the argument is the value
    # already in force rather than something to apply by hand.
    return solve_pnp(prof, default_species()).conductance * 1e12


spread = parameter_range(conductance_at, "permeation.diffusion_scale",
                         [0.4, 0.7, 1.0, 1.5, 2.0],
                         what="single-channel conductance, pS")
print(spread.summary())
print("\nThis is a PARAMETER range propagated from an unmeasured input.")
print("It is not a confidence interval and the class will not call it one.")

## Seeing it move

`piezo1.render.flux` turns the computed current into an animation time base. A
channel passes about 10⁷ ions per second, so any watchable stream runs roughly
a millionfold slow — and the number is computed from the solver's own output
rather than chosen to look good, so the display can state it.

In [ ]:
from piezo1.render.flux import ion_rate, timebase

current_pA = result.current * 1e12
tb = timebase(current_pA)
print(f"{tb.ions_per_second:.3e} ions per second")
print(f"slowdown for a watchable stream: {tb.slowdown:,.0f}x")
print(f"\n{tb.statement()}")